#### > Autograd(자동미분)
: PyTorch 에서 연산과정을 추적하고 역전파를 통해 미분값을 자동으로 계산하는 기능
 - **미분 계산 자동화**
 - **기울기 계산** : 각 입력값과 모델의 가중치가 손실함수에 미치는 영향을 계산
 - **모델 학습 지원** : 계산한 기울기를 optimizer 에 전달하여 가중치를 업데이트

In [1]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

##### > Autograd 의 기본 흐름

1. 미분할 텐서에 requires_grad = True 설정
2. 해당 텐서를 사용하여 연산 수행
3. 결과 텐서에 backward() 실행. -> 거꾸로 올라가면서 미분
4. 원본 텐서의 grad 속성에서 미분값 확인 가능

#### 1. requires_grad
- True : 해당 텐서에 대한 연산을 추적하여 기울기를 계산할 수 있도록 설정.
- False : 기본값, 연산을 추적하지 않음
- 일반적으로 모델이 학습해야 하는 가중치(weight)와 편향(bias)에 True를 설정함

In [2]:
x = torch.tensor(1.0, requires_grad= True)
print(x)

tensor(1., requires_grad=True)


In [ ]:
x = torch.tensor(1.0)   # requires_grad= False
print(x)
print(x.requires_grad)

x.requires_grad=True    # 수정 가능
print(x.requires_grad)

x.requires_grad=False   # 재수정(turn off) 가능
print(x.requires_grad)

tensor(1.)
False
True
False


In [ ]:
# 1. 텐서 생성
x = torch.tensor([1.0], requires_grad=True)

# 2. 함수 정의
y = x**2
print(y) # tensor([1.], grad_fn=<PowBackward0>) -> 제곱을 했음을 추적

# 3. 역전파 실행
y.backward()

# 4. 결과와 미분값(기울기, grad)
print('y: ', y)                # y:  tensor([1.], grad_fn=<PowBackward0>)
print('x의 기울기: ', x.grad)   # x의 기울기:  tensor([2.])

tensor([1.], grad_fn=<PowBackward0>)
y:  tensor([1.], grad_fn=<PowBackward0>)
x의 기울기:  tensor([2.])


In [ ]:
x = torch.tensor(1.0, requires_grad=True)
x.requires_grad=False
    # transfer learning 할 때 사용
    # 내가 만들어 놓은 학습된 모델에서 일부분만 학습시키고 싶을 때 나머지 부분의 추적을 off
y = x**2
print(y)
# y.backward() error

tensor(1.)


In [ ]:
x = torch.tensor([2., 4], requires_grad=True)
y = x * 3
print('x.requires_grad:', x.requires_grad)
print('y.requires_grad:', y.requires_grad)
print('x.is_leaf:', x.is_leaf) # leaf 여부 
print('y.is_leaf:', y.is_leaf)
print('y.grad_fn:', y.grad_fn) # 어떤 연산을 통해 y 를 도출했는지 설명 (MulBackward0 : 곱셈 연산을 했음)

x.requires_grad: True
y.requires_grad: True
x.is_leaf: True
y.is_leaf: False
y.grad_fn: <MulBackward0 object at 0x0000022BB0D28C10>


#### 2. 연쇄법칙(Chain Rule)
: 함수가 여러 연산으로 연결되어 있으면 각 연산의 미분값을 연쇄적으로 곱하여 최종 기울기를 계산함

In [ ]:
x = torch.tensor(1., requires_grad=True)
y = x**2
print(y)

z = 3 * y
print(z)

z.backward()
print(x.grad) # 최종 미분값 할당은 leaf tensor 인 x 에만
print(y.grad) # None

tensor(1., grad_fn=<PowBackward0>)
tensor(3., grad_fn=<MulBackward0>)
tensor(6.)
None


C:\Users\mega\AppData\Local\Temp\ipykernel_9836\4062033234.py:10: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:493.)
  print(y.grad)
